In [1]:
# Setup and connection
import sqlite3
import pandas as pd

conn = sqlite3.connect("financials.db")

query = """
SELECT COUNT(*) FROM financials;
"""
pd.read_sql(query, conn)

,COUNT(*)
0,700


In [2]:
# distinct-dates check
query = """
SELECT typeof(date) AS dtype,
       MIN(date) AS first_date,
       MAX(date) AS last_date,
       COUNT(DISTINCT date) AS distinct_dates,
       COUNT(DISTINCT strftime('%Y-%m', date)) AS distinct_months
FROM financials;
"""
pd.read_sql(query, conn)

,dtype,first_date,last_date,distinct_dates,distinct_months
0,text,2013-09-01 00:00:00,2014-12-01 00:00:00,16,16


In [3]:
# month-over-month growth
query1 = """
WITH monthly AS (
    SELECT strftime('%Y-%m', date) AS month,
           SUM(sales)  AS total_sales,
           SUM(profit) AS total_profit
    FROM financials
    GROUP BY 1
),
with_prev AS (
    SELECT month,
           total_sales,
           total_profit,
           LAG(total_sales) OVER (ORDER BY month) AS prev_sales
    FROM monthly
)
SELECT month,
       total_sales,
       prev_sales,
       ROUND(100.0 * (total_sales - prev_sales) / prev_sales, 2) AS mom_growth_pct
FROM with_prev
ORDER BY month;
"""
df1 = pd.read_sql(query1, conn)
df1

,month,total_sales,prev_sales,mom_growth_pct
0,2013-09,4484000.03,NaN,NaN
1,2013-10,9295611.10,4484000.03,107.31
2,2013-11,7267203.30,9295611.10,-21.82
3,2013-12,5368441.08,7267203.30,-26.13
4,2014-01,6607761.68,5368441.08,23.09
5,2014-02,7297531.39,6607761.68,10.44
6,2014-03,5586859.87,7297531.39,-23.44
7,2014-04,6964775.07,5586859.87,24.66
8,2014-05,6210211.06,6964775.07,-10.83
9,2014-06,9518893.82,6210211.06,53.28


In [4]:
# top 3 products per segment
query2 = """
WITH product_profit AS (
    SELECT segment,
           product,
           SUM(profit) AS total_profit
    FROM financials
    GROUP BY segment, product
),
ranked AS (
    SELECT segment,
           product,
           total_profit,
           RANK() OVER (PARTITION BY segment ORDER BY total_profit DESC) AS rnk
    FROM product_profit
)
SELECT segment, rnk, product, ROUND(total_profit, 0) AS total_profit
FROM ranked
WHERE rnk <= 3
ORDER BY segment, rnk;
"""
df2 = pd.read_sql(query2, conn)
df2

,segment,rnk,product,total_profit
0,Channel Partners,1,Paseo,331838.0
1,Channel Partners,2,Amarilla,230069.0
2,Channel Partners,3,VTT,219766.0
3,Enterprise,1,Montana,-31096.0
4,Enterprise,2,Paseo,-81740.0
5,Enterprise,3,Velo,-84763.0
6,Government,1,Paseo,3057291.0
7,Government,2,Amarilla,2208302.0
8,Government,3,VTT,1840654.0
9,Midmarket,1,Paseo,258739.0


In [5]:
# 3-month moving average
query3 = """
WITH monthly AS (
    SELECT strftime('%Y-%m', date) AS month,
           SUM(sales) AS total_sales
    FROM financials
    GROUP BY 1
)
SELECT month,
       total_sales,
       ROUND(AVG(total_sales) OVER (
           ORDER BY month
           ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
       ), 0) AS moving_avg_3m,
       COUNT(*) OVER (
           ORDER BY month
           ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
       ) AS months_in_window
FROM monthly
ORDER BY month;
"""
df3 = pd.read_sql(query3, conn)
df3

,month,total_sales,moving_avg_3m,months_in_window
0,2013-09,4484000.03,4484000.0,1
1,2013-10,9295611.10,6889806.0,2
2,2013-11,7267203.30,7015605.0,3
3,2013-12,5368441.08,7310418.0,3
4,2014-01,6607761.68,6414469.0,3
5,2014-02,7297531.39,6424578.0,3
6,2014-03,5586859.87,6497384.0,3
7,2014-04,6964775.07,6616389.0,3
8,2014-05,6210211.06,6253949.0,3
9,2014-06,9518893.82,7564627.0,3
